# 📘 Notebook 11: Topic Modelling

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module D: Meaning as Vectors · Notebook 2 of 2 in this module · (11 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a **topic model** is and when you would reach for one
- Describe a document as a **mixture of topics** and a topic as a **weighted list of words**
- Factorise a document-term matrix with **NMF**, by hand and with scikit-learn
- Explain the generative story behind **LDA** and fit an LDA model
- Read, name and sanity-check the topics a model finds, using labels the model never saw
- Discuss the **limits** of topic models: the choice of the number of topics, instability, and the need for human judgement

> **Prerequisites:** Notebooks 08 and 10. Matrix multiplication with NumPy, from *From Python to Deep Learning & Fine-Tuning*, Notebook 05. Unsupervised learning in general is introduced in Notebook 11 of that course.
>
> 🔭 **Why this notebook matters:** in Notebook 09 we classified documents into categories that somebody had defined and labelled in advance. Most of the text in the world comes with no labels at all: a mailbox of customer complaints, ten years of meeting minutes, a hundred thousand survey answers. Topic modelling is how you get a first map of such a collection. It answers the question *"what is in here?"* before you know what to look for.

> ℹ️ **Setup note.** Run this cell once.

> 📦 **Libraries used in this notebook.**
>
> - **scikit-learn**, introduced in Notebook 08, provides ready-made implementations of the two topic models of this notebook, `NMF` and `LatentDirichletAllocation`.
> - **NumPy** is used for the factorisation we write by hand, **Matplotlib** for the plots, and **NLTK** for the Brown corpus.
>
> The cell below installs what is needed. In Google Colab these libraries are already present and the cell finishes in a moment. On your own computer it may take a minute the first time.

In [ ]:
%pip install -q nltk numpy matplotlib scikit-learn

In [ ]:
import nltk
nltk.download("brown", quiet=True)

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## 1. The idea

### Intuition first
Pick up a newspaper. One article is about a football transfer: it is mostly *sport* with some *money*. Another reports on a hospital budget: part *health*, part *politics*, part *money*. Nobody would file these articles under a single heading. Each one **mixes** several themes in different proportions.

And how do you recognise a theme? By its vocabulary. *goal*, *match*, *striker* and *league* tend to turn up together. So do *patient*, *ward*, *nurse* and *treatment*.

A **topic model** rests on exactly these two ideas:

1. a **topic** is a group of words that tend to occur together, and
2. a **document** is a mixture of a few topics.

Given only the documents, the model must discover both the topics and the mixtures. It is the distributional hypothesis of Notebook 10 again, one level up: there we looked at which words share a small window, here at which words share a document.

### Formal definition
Start from the document-term matrix $V$ of Notebook 08, with one row per document and one column per word. A topic model with $k$ topics looks for two smaller matrices:

- $W$, of size documents × $k$: how much of each **topic** is in each **document**
- $H$, of size $k$ × words: how much each **word** belongs to each **topic**

such that their product approximately reproduces the original:

$$V \approx W \times H$$

Thousands of word columns are explained through a handful of topics in the middle.

## 2. A factorisation by hand

Six tiny documents on two obvious themes make the idea concrete.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

mini_documents = [
    "the team won the match with a late goal",
    "the striker scored a goal in the final match",
    "the coach praised the team after the match",
    "add the garlic and the onion to the pan",
    "fry the onion in the pan with olive oil",
    "the goal of the chef is a rich garlic sauce with olive oil",
]

mini_vectoriser = CountVectorizer(stop_words="english")
V = mini_vectoriser.fit_transform(mini_documents).toarray().astype(float)
mini_words = list(mini_vectoriser.get_feature_names_out())

print(V.shape, "documents x words")
print(mini_words)

### Formal definition
**Non-negative matrix factorisation (NMF)** looks for $W$ and $H$ whose entries are all **zero or positive**, making the difference between $V$ and $W \times H$ as small as possible. The restriction to non-negative numbers is what makes the result readable: a document can contain some amount of a topic or none of it, but never a negative amount.

A classic algorithm starts from random matrices and repeats two multiplicative updates. Each one scales the entries of a matrix up where the reconstruction is too small and down where it is too large.

$$H \leftarrow H \odot \frac{W^\top V}{W^\top W H} \qquad\qquad W \leftarrow W \odot \frac{V H^\top}{W H H^\top}$$

Here $\odot$ and the fraction bar mean element-by-element multiplication and division.

In [ ]:
rng = np.random.default_rng(0)
topics = 2
W = rng.random((V.shape[0], topics))
H = rng.random((topics, V.shape[1]))

for step in range(201):
    H *= (W.T @ V) / (W.T @ W @ H + 1e-9)
    W *= (V @ H.T) / (W @ H @ H.T + 1e-9)
    if step % 50 == 0:
        error = np.linalg.norm(V - W @ H)
        print(f"step {step:>3}  reconstruction error {error:.3f}")

The error falls and settles. Now read the two matrices. Each row of $H$ is a topic: we list its words from the heaviest down.

In [ ]:
for topic in range(topics):
    order = np.argsort(-H[topic])
    print(f"Topic {topic}:", [mini_words[i] for i in order[:6]])

print()
print("Share of each topic in each document:")
shares = W / W.sum(axis=1, keepdims=True)
for document, row in zip(mini_documents, shares):
    print(f"  {row[0]:.2f}  {row[1]:.2f}   {document}")

Two topics have appeared, one about football and one about cooking, and the first five documents are assigned almost entirely to one or the other. The program was told nothing about sport or food. It found that some words keep turning up in the same documents.

Look at the last document. It is a cooking sentence that happens to contain the word *goal*, and the model gives it a small share of the football topic. That is the mixture idea at work, and also a first warning: the model sees words, and it does not know that *goal* is being used in a different sense.

## 3. Topics in a real collection

The Brown corpus has 500 documents of about 2,000 words each, sampled from very different kinds of writing: news, religion, science, government reports, hobbies, and several kinds of fiction. Each document carries a genre label. **We will not show the labels to the model.** We keep them aside to check its work afterwards.

In [ ]:
from nltk.corpus import brown

file_ids = brown.fileids()
documents = [" ".join(brown.words(file_id)) for file_id in file_ids]
genres = [brown.categories(file_id)[0] for file_id in file_ids]

print(len(documents), "documents")
print(Counter(genres).most_common())

In [ ]:
from sklearn.decomposition import NMF

vectoriser = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5, max_features=4000, token_pattern=r"[A-Za-z]{3,}")
matrix = vectoriser.fit_transform(documents)
words = vectoriser.get_feature_names_out()
print("Matrix:", matrix.shape)

number_of_topics = 10
nmf = NMF(n_components=number_of_topics, init="nndsvd", random_state=0, max_iter=200)
document_topics = nmf.fit_transform(matrix)       # W: documents x topics
topic_words = nmf.components_                     # H: topics x words

def show_topics(topic_words, how_many=10):
    for topic, row in enumerate(topic_words):
        order = np.argsort(-row)[:how_many]
        print(f"Topic {topic}: " + ", ".join(words[i] for i in order))

show_topics(topic_words)

Read the ten lists and try to give each one a name. Several are unmistakable. The option `max_df=0.5` removed words that occur in more than half of the documents, which no topic could claim, and `min_df=5` removed words too rare to form a pattern.

A topic model gives you lists of words. The **names are yours**. Naming a topic is an act of interpretation, and a different reader might choose differently.

### A document is a mixture
Let us look at the topic shares of three individual documents.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
for axis, index in zip(axes, [0, 150, 420]):
    shares = document_topics[index] / (document_topics[index].sum() + 1e-9)
    axis.bar(range(number_of_topics), shares)
    axis.set_title(f"{file_ids[index]}  (genre: {genres[index]})", fontsize=10)
    axis.set_xticks(range(number_of_topics))
    axis.set_xlabel("topic")
axes[0].set_ylabel("share of the document")
plt.tight_layout()
plt.show()

for index in [0, 150, 420]:
    print(f"{file_ids[index]} ({genres[index]}):", documents[index][:110], "...")

Each document is dominated by one or two topics and has small shares of others. Compare each bar chart with the opening words of the document and with the word list of its strongest topic.

### Checking against the labels we hid
Do the discovered topics correspond to anything real? For each genre, we find the topic that is strongest on average across the documents of that genre.

In [ ]:
genre_names = sorted(set(genres))
genre_array = np.array(genres)
shares = document_topics / (document_topics.sum(axis=1, keepdims=True) + 1e-9)

print(f"{'genre':<17} {'top topic':>9} {'share':>6}   heaviest words of that topic")
for genre in genre_names:
    average = shares[genre_array == genre].mean(axis=0)
    top = int(np.argmax(average))
    top_words = ", ".join(words[i] for i in np.argsort(-topic_words[top])[:5])
    print(f"{genre:<17} {top:>9} {average[top]:>6.2f}   {top_words}")

The model never saw the genre labels, and yet its topics line up with them in a sensible way. Genres with a distinctive vocabulary get a topic of their own. The fiction genres tend to share a topic, because adventure stories, romances and mysteries are told with much the same everyday words. That is a fair summary of the collection, reached without supervision.

## 4. A probabilistic view: LDA

### Intuition first
NMF is a piece of arithmetic. **Latent Dirichlet Allocation (LDA)** arrives at a similar result from a story about how documents are written. Imagine an author with a fixed set of topics, each of which is a bag of words with its own favourite words. To write a document, the author:

1. decides on a mixture of topics for this document, say 70% sport and 30% money;
2. for every word to be written, picks a topic according to that mixture,
3. and then draws a word from the bag of that topic.

Real writers do no such thing, of course. But if we *pretend* the documents were produced this way, we can ask: which topics and which mixtures would make the documents we actually have most probable? Answering that question is called **inference**, and it is what fitting an LDA model does.

### The story, simulated
Let us play the author. We define two topics by hand and generate documents from them.

In [ ]:
rng = np.random.default_rng(1)

story_vocabulary = ["goal", "match", "team", "striker", "price", "bank", "market", "profit"]
story_topics = np.array([
    [0.30, 0.30, 0.25, 0.15, 0.00, 0.00, 0.00, 0.00],     # a topic about sport
    [0.00, 0.00, 0.00, 0.00, 0.25, 0.25, 0.25, 0.25],     # a topic about money
])

def write_document(length=10):
    mixture = rng.dirichlet([0.5, 0.5])                    # step 1: a mixture of the two topics
    written = []
    for i in range(length):
        topic = rng.choice(2, p=mixture)                   # step 2: pick a topic for this word
        word = rng.choice(story_vocabulary, p=story_topics[topic])   # step 3: draw a word from it
        written.append(word)
    return mixture, " ".join(written)

for i in range(5):
    mixture, text = write_document()
    print(f"sport {mixture[0]:.2f}, money {mixture[1]:.2f}   {text}")

The line `rng.dirichlet([0.5, 0.5])` gives the method its name. A **Dirichlet distribution** produces random mixtures, lists of positive numbers that add up to 1. With small parameters, as here, most of the mixtures it produces are lopsided: mostly one topic. That builds in the assumption that a typical document is about only a few things.

### Formal definition
LDA assumes each document $d$ has a topic mixture $\theta_d$ drawn from a Dirichlet distribution, and each topic $t$ has a word distribution $\beta_t$, also drawn from a Dirichlet distribution. Each word is generated by choosing a topic from $\theta_d$ and then a word from the corresponding $\beta_t$. Fitting the model means estimating the $\theta$ and $\beta$ that best explain the observed words.

Because LDA is a model of word **counts**, it is fitted on raw counts and not on TF-IDF weights.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

count_vectoriser = CountVectorizer(stop_words="english", min_df=5, max_df=0.5, max_features=4000, token_pattern=r"[A-Za-z]{3,}")
count_matrix = count_vectoriser.fit_transform(documents)

lda = LatentDirichletAllocation(n_components=number_of_topics, random_state=0, max_iter=15)
lda_document_topics = lda.fit_transform(count_matrix)

show_topics(lda.components_)

Compare these with the NMF topics above. Some themes appear in both models, perhaps split or merged differently. Others appear in only one. Neither set is "the" right answer. They are two summaries of the same collection, made under different assumptions.

> ⚠️ **Common pitfalls**
>
> - Reading topics as facts. A topic is a pattern of co-occurrence that a particular algorithm found with particular settings. Always read some of the **documents** behind a topic before you believe its word list.
> - Skipping the cleaning. Without `stop_words`, `max_df` and `min_df`, the first topics are made of *the*, *said*, *would* and page headers. Topic models are extremely sensitive to preprocessing.
> - Giving LDA TF-IDF weights, or documents of two or three words. LDA needs counts, and enough words per document for co-occurrence to mean something.
> - Running the model once. The next section shows why.

## 5. How many topics, and how stable?

The number of topics $k$ is a choice we make. There is no correct value, only values that are more or less useful for a purpose. With few topics you get broad themes. With many you get fine distinctions, and eventually topics made of noise.

In [ ]:
for k in [3, 6]:
    model = NMF(n_components=k, init="nndsvd", random_state=0, max_iter=200).fit(matrix)
    print(f"--- {k} topics")
    show_topics(model.components_, how_many=7)

One measurement we can make is how well $k$ topics reconstruct the matrix. The error can only go down as $k$ grows, so we do not look for the minimum. We look for the point where adding topics stops helping much.

In [ ]:
values = [2, 4, 6, 8, 10, 15, 20]
errors = []
for k in values:
    model = NMF(n_components=k, init="nndsvd", random_state=0, max_iter=100).fit(matrix)
    errors.append(model.reconstruction_err_)

plt.figure(figsize=(7, 4))
plt.plot(values, errors, marker="o")
plt.title("NMF reconstruction error against the number of topics")
plt.xlabel("Number of topics (k)")
plt.ylabel("Reconstruction error")
plt.grid(True)
plt.show()

The curve falls smoothly, with no sharp corner to tell us where to stop. That is the honest state of affairs: the data do not dictate $k$. In practice people try a few values and keep the one whose topics they can interpret and use.

There is a second source of uncertainty. Both algorithms start from random numbers, and different starting points can lead to different topics.

In [ ]:
runs = []
for seed in [0, 1, 2]:
    model = NMF(n_components=number_of_topics, init="random", random_state=seed, max_iter=200).fit(matrix)
    runs.append([set(words[i] for i in np.argsort(-row)[:10]) for row in model.components_])

print("For each topic of run 0: the most words it shares (out of 10) with any topic of run 1 and of run 2")
for topic, top_words in enumerate(runs[0]):
    with_run_1 = max(len(top_words & other) for other in runs[1])
    with_run_2 = max(len(top_words & other) for other in runs[2])
    print(f"  topic {topic}: {with_run_1:>2} and {with_run_2:>2}")

Topics that come back nearly unchanged in every run are a solid feature of the collection. Topics that share only a few words between runs are fragile, and conclusions should not be built on them.

> 🧠 **How to use a topic model well.** Treat it as an instrument for **exploration**, like a first look through a telescope. It tells you where to point your attention. It is excellent for getting an overview of a large unlabelled collection, for finding documents on a theme, and for suggesting categories worth labelling properly. It is poor evidence for a claim. If a decision depends on it, read the documents.

## 6. Where Module D leaves us

| | Bag of words + TF-IDF | Word embeddings | Topic models |
|---|---|---|---|
| Unit represented | a document | a word | a document **and** its themes |
| Size of the vector | the whole vocabulary | a few hundred numbers | a handful of topic shares |
| Needs labelled data | no | no | no |
| Knows that two words are related | no | yes | yes, if they share a topic |
| Uses word order | no | no | no |
| One word, several meanings | no | no | partly: a word can belong to several topics |

Module D replaced isolated symbols with geometry. Words became points whose distances reflect meaning, and documents became mixtures of themes. All of it was learned from raw text, with no labels.

One row has read "no" since the very first notebook: **word order**. Every representation so far would give *"the dog bit the man"* and *"the man bit the dog"* the same description. The next module finally takes order seriously.

---
## ✏️ Exercises

### Exercise 1 (A new document)
The hand-made factorisation of Section 2 gave us a matrix `H` of two topics. Given a **new** document, we can find its topic shares by keeping `H` fixed and updating only its own row of `W`. Do this for the sentence `"the chef scored a goal with garlic"`: vectorise it with `mini_vectoriser.transform`, start from a random row, apply the update rule for `W` 200 times, and print the shares.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
new_V = mini_vectoriser.transform(["the chef scored a goal with garlic"]).toarray().astype(float)
new_W = np.random.default_rng(0).random((1, topics))

for step in range(200):
    new_W *= (new_V @ H.T) / (new_W @ H @ H.T + 1e-9)

print(dict(zip(mini_words, new_V[0].astype(int))))
print("Topic shares:", np.round(new_W / new_W.sum(), 2))
```

*The sentence mixes vocabulary from both themes, and its shares say so. Keeping the topics fixed and solving only for the mixture of a new document is what the `transform` method of scikit-learn's models does.*
</details>

### Exercise 2 (Name the topics)
Fit an NMF model with **15** topics on `matrix` and print the top 8 words of each. Write down a name for each topic. Are there topics you cannot name? Compared with the 10-topic model, which themes have split in two?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
nmf_15 = NMF(n_components=15, init="nndsvd", random_state=0, max_iter=200).fit(matrix)
show_topics(nmf_15.components_, how_many=8)
```

*There is no answer key for this exercise, which is the point of it. With more topics, broad themes divide into narrower ones, and one or two lists usually resist naming. The usefulness of a topic model is judged by a person, and two people may judge differently.*
</details>

### Exercise 3 (Find documents by topic)
Choose a topic from the 10-topic NMF model whose word list you find clear. Print the file id, genre and first 100 characters of the five documents in which that topic has the **largest share**. Do the documents fit the name you gave the topic?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
chosen = 1
print("Topic words:", ", ".join(words[i] for i in np.argsort(-topic_words[chosen])[:8]), "\n")

for index in np.argsort(-shares[:, chosen])[:5]:
    print(f"{shares[index, chosen]:.2f}  {file_ids[index]}  {genres[index]:<12} {documents[index][:100]}")
```

*Going from the word list back to the documents is the essential check. It is also the practical use: this is a search engine that needs no query, only a theme.*
</details>

### Exercise 4 (Topics as features)
The topic shares are a compact description of each document, just 10 numbers. Are they enough to predict the genre? Train a logistic regression to tell the genres `news`, `religion`, `romance` and `government` apart, once using the 10 NMF topic weights and once using the full TF-IDF matrix. Compare with five-fold cross-validation.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

chosen_genres = ["news", "religion", "romance", "government"]
mask = np.isin(genre_array, chosen_genres)
labels = genre_array[mask]

with_topics = cross_val_score(LogisticRegression(C=100, max_iter=5000), document_topics[mask], labels, cv=5).mean()
with_words = cross_val_score(LogisticRegression(C=100, max_iter=5000), matrix[mask], labels, cv=5).mean()

print(f"{mask.sum()} documents")
print(f"10 topic weights:        {with_topics:.1%}")
print(f"{matrix.shape[1]:,} TF-IDF weights:   {with_words:.1%}")
```

*Ten numbers per document do almost as well as four thousand. (The setting `C=100` weakens the regularisation of the classifier, which is needed here because the feature values are small.) Reducing documents to a few dense features before classifying them is a common use of topic models, and of the embeddings of Notebook 10.*
</details>

### Exercise 5 (Do NMF and LDA agree? A little harder)
For each NMF topic, find the LDA topic that is most similar to it. Measure similarity as the number of words the two topics share among their top 15. Note that the two models use the same vocabulary in the same order. Print each NMF topic, its best LDA match and the number of shared words.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def top_word_sets(components, how_many=15):
    return [set(words[i] for i in np.argsort(-row)[:how_many]) for row in components]

nmf_sets = top_word_sets(topic_words)
lda_sets = top_word_sets(lda.components_)

for topic, nmf_set in enumerate(nmf_sets):
    overlaps = [len(nmf_set & lda_set) for lda_set in lda_sets]
    best = int(np.argmax(overlaps))
    print(f"NMF topic {topic} ~ LDA topic {best}: {overlaps[best]:>2} shared words   {sorted(nmf_set & lda_sets[best])[:6]}")
```

*Where two quite different algorithms find the same topic, we have more reason to think it reflects the collection and not the method. Where they disagree, be cautious. Agreement between independent methods is one of the few forms of validation available when there are no labels.*
</details>

## 🔑 Key takeaways

- A **topic model** discovers themes in an unlabelled collection. A **topic** is a weighted list of words, and a **document** is a mixture of topics.
- **NMF** factorises the document-term matrix into non-negative document-topic and topic-word matrices: $V \approx W \times H$.
- **LDA** reaches a similar result from a generative story: choose a mixture, then for each word choose a topic and draw a word from it.
- Topics must be **read and named by a person**, and checked against the documents behind them.
- The **number of topics** is a choice, and results vary with the random start. Trust the topics that are **stable**.
- Topic models are tools for **exploring** a collection, not evidence for claims about it.

### 🏁 You have completed Module D: Meaning as Vectors!
Words are now points in a space where neighbours are related and directions carry meaning, and documents are mixtures of themes, all learned from unlabelled text.

**Next:** *Module E: Neural Networks for Text*, starting with *Notebook 12: LSTMs and Text Generation*. There we take the vectors of this module and feed them, in order, into networks that read a sentence one word at a time.

---
*End of Module D.*

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*